## 1. Setup Colab

**Runtime > Modifier le type d'exécution > GPU** avant de commencer.

Remplacez `OWNER` par votre nom d'utilisateur GitHub.

In [ ]:
# Cloner le dépôt puis s'y placer.
!git clone https://github.com/OWNER/cross-domain-visual-anomaly-detection.git
%cd cross-domain-visual-anomaly-detection

In [ ]:
# Installer le package (PyTorch est déjà présent sur Colab).
!pip -q install -e .

In [ ]:
# Vérifier le GPU.
import torch

print(
    "CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
# (Optionnel) Monter Google Drive pour sauvegarder checkpoints et métriques.
# from google.colab import drive
# drive.mount('/content/drive')


## 2. Expériences inter-domaines

Le script exécute le même pipeline sur les trois domaines. Par défaut il
génère des données **synthétiques** ; pour le réel, téléchargez MVTec /
RSNA / xView2 et adaptez `scripts/run_cross_domain.py` (chemins des
datasets réels).

In [ ]:
!python scripts/run_cross_domain.py --epochs 10 --image-size 128

## 3. Tableau comparatif

In [ ]:
import json
import pandas as pd

data = json.loads(open("reports/metrics/cross_domain.json").read())
pd.DataFrame(data)

## 4. Analyse

Comparez AUROC / F1 entre domaines **avec prudence** : les datasets et
types d'anomalie diffèrent. Notez ce qui est commun (le pipeline) vs
adapté (splits par patient / par zone).

## Récupérer les artefacts en local

Les modèles/métriques sont dans `models/` et `reports/`. Copiez-les vers
Drive (ci-dessus) ou téléchargez-les, puis committez-les **depuis votre
machine** (ne poussez pas de gros checkpoints ; préférez Drive).

In [ ]:
# Copier les artefacts vers Drive (si monté).
# !mkdir -p /content/drive/MyDrive/anomaly_artifacts
# !cp -r models reports /content/drive/MyDrive/anomaly_artifacts/


In [ ]:
# Commande git à exécuter EN LOCAL après récupération des artefacts :
print(r"""git add reports/ docs/ && \
git commit -m "docs(report): add real experiment metrics from Colab" && \
git push origin main""")

## Reprise après interruption

Colab peut se déconnecter. Les checkpoints sont sauvegardés dans `models/`.
Après reconnexion, ré-exécutez le Setup puis relancez la cellule
d'entraînement : un checkpoint existant sert de point de reprise / base
d'évaluation (l'entraînement complet reste à relancer si non terminé).